In [ ]:
import os
import ctypes
import concurrent.futures
from pathlib import Path
from tqdm import tqdm

# Windows DeleteFileW (kernel-level delete)
DeleteFileW = ctypes.windll.kernel32.DeleteFileW
DeleteFileW.argtypes = [ctypes.c_wchar_p]
DeleteFileW.restype = ctypes.c_bool

def fast_delete(path: Path):
    try:
        DeleteFileW(str(path))
    except Exception:
        try:
            os.remove(path)
        except:
            pass

def delete_directory_fast(directory: str, workers: int = 16):
    directory = Path(directory)

    if not directory.exists():
        print(f"Directory does not exist: {directory}")
        return

    print(f"Deleting all files in: {directory}")
    print(f"Using {workers} workers...")

    # Use scandir for extremely fast enumeration
    files = []
    stack = [directory]

    while stack:
        current = stack.pop()
        with os.scandir(current) as it:
            for entry in it:
                if entry.is_file(follow_symlinks=False):
                    files.append(Path(entry.path))
                elif entry.is_dir(follow_symlinks=False):
                    stack.append(Path(entry.path))

    total_files = len(files)
    print(f"Total files detected: {total_files}")

    # Multi-threaded deletion with progress bar
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as exe:
        futures = [exe.submit(fast_delete, f) for f in files]

        with tqdm(total=total_files, desc="Deleting", unit="file") as pbar:
            for _ in concurrent.futures.as_completed(futures):
                pbar.update(1)

    print("File deletion complete.")
    print("Removing empty directories...")

    # Remove empty dirs bottom-up
    for root, dirs, _ in os.walk(directory, topdown=False):
        for d in dirs:
            try:
                os.rmdir(Path(root) / d)
            except:
                pass

    print("Directory cleanup finished.")

# Run
delete_directory_fast(
    directory=r'G:\Thesis\ImageRetrieval\Professions_125k\Male_Administrative_Assistant',
    workers=8 #32
)

Deleting all files in: G:\Thesis\ImageRetrieval\Professions_125k\Male_Administrative_Assistant
Using 8 workers...
Total files detected: 24986


Deleting:  67%|██████▋   | 16765/24986 [17:04<25:02,  5.47file/s]  

In [ ]:
import os
import ctypes
import concurrent.futures
from pathlib import Path
from tqdm import tqdm

# Windows API functions
DeleteFileW = ctypes.windll.kernel32.DeleteFileW
DeleteFileW.argtypes = [ctypes.c_wchar_p]
DeleteFileW.restype = ctypes.c_bool

SetFileAttributesW = ctypes.windll.kernel32.SetFileAttributesW
SetFileAttributesW.argtypes = [ctypes.c_wchar_p, ctypes.c_uint32]
SetFileAttributesW.restype = ctypes.c_bool

FILE_ATTRIBUTE_NORMAL = 0x80

def fast_delete(path: Path):
    try:
        path_str = str(path)
        SetFileAttributesW(path_str, FILE_ATTRIBUTE_NORMAL)  # Clear attributes first
        DeleteFileW(path_str)
    except Exception:
        try:
            os.remove(path)
        except:
            pass

def delete_directory_fast(directory: str, workers: int = 64):
    directory = Path(directory)

    if not directory.exists():
        print(f"Directory does not exist: {directory}")
        return

    print(f"Deleting all files in: {directory}")
    print(f"Using {workers} workers...")

    # Fast file enumeration
    files = []
    stack = [directory]

    # while stack:
    #     current = stack.pop()
    #     with os.scandir(current) as it:
    #         for entry in it:
    #             if entry.is_file(follow_symlinks=False):
    #                 print("New file found")
    #                 files.append(Path(entry.path))
    #             elif entry.is_dir(follow_symlinks=False):
    #                 print("New dir found")
    #                 stack.append(Path(entry.path))

    stop = False

    with tqdm(desc="Files discovered", unit="files", mininterval=0.0) as pbar:
        while stack and not stop:
            current = stack.pop()
            with os.scandir(current) as it:
                for entry in it:
                    if entry.is_file(follow_symlinks=False):
                        files.append(Path(entry.path))
                        pbar.update(1)
                    elif entry.is_dir(follow_symlinks=False):
                        stack.append(Path(entry.path))

                    if len(files) % 500_000 == 0 and len(files) > 0:
                        print(f"Discovered {len(files)} files so far...")
                        stop = True
                        break  # For testing purposes, remove this line for full deletion

    total_files = len(files)
    print(f"Total files detected: {total_files}")

    # Multi-threaded deletion with chunking
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as exe:
        with tqdm(total=total_files, desc="Deleting", unit="file") as pbar:
            for _ in exe.map(fast_delete, files, chunksize=500):
                pbar.update(1)

    print("File deletion complete.")
    
    # Optional: Skip if you don't need empty dir cleanup
    print("Removing empty directories...")
    for root, dirs, _ in os.walk(directory, topdown=False):
        for d in dirs:
            try:
                os.rmdir(Path(root) / d)
            except:
                pass

    print("Directory cleanup finished.")

# Run with SSD-optimized settings
delete_directory_fast(
    directory=r'G:\Thesis\ImageRetrieval\Professions_20k_DELETE',
    workers=64  # High worker count for SSD parallel I/O
)

Deleting all files in: G:\Thesis\ImageRetrieval\Professions_20k_DELETE
Using 64 workers...


Files discovered: 5files [00:00, 999.68files/s] 

Files discovered: 271263files [02:04, 940.87files/s] 

In [ ]:
import os 
import ctypes 
import concurrent.futures 
from pathlib import Path 
from tqdm import tqdm 
from collections import deque
 
# Windows API functions 
DeleteFileW = ctypes.windll.kernel32.DeleteFileW 
DeleteFileW.argtypes = [ctypes.c_wchar_p] 
DeleteFileW.restype = ctypes.c_bool 
 
SetFileAttributesW = ctypes.windll.kernel32.SetFileAttributesW 
SetFileAttributesW.argtypes = [ctypes.c_wchar_p, ctypes.c_uint32] 
SetFileAttributesW.restype = ctypes.c_bool 
 
FILE_ATTRIBUTE_NORMAL = 0x80 
 
def fast_delete(path_str: str):
    """Delete using string path directly to avoid Path object overhead"""
    try: 
        SetFileAttributesW(path_str, FILE_ATTRIBUTE_NORMAL)
        DeleteFileW(path_str) 
    except Exception: 
        try: 
            os.remove(path_str) 
        except: 
            pass 
 
def delete_directory_fast(directory: str, workers: int = 64, batch_size: int = 50000): 
    directory = Path(directory) 
 
    if not directory.exists(): 
        print(f"Directory does not exist: {directory}") 
        return 
 
    print(f"Deleting all files in: {directory}") 
    print(f"Using {workers} workers with batch size {batch_size}...") 
 
    # Use deque for better performance than list
    stack = deque([directory])
    files_batch = []
    total_deleted = 0
    total_discovered = 0
 
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as exe:
        with tqdm(desc="Discovering", unit="files", position=0) as discover_pbar, \
             tqdm(desc="Deleting", unit="files", position=1) as delete_pbar:
            while stack:
                current = stack.pop()
                
                try:
                    with os.scandir(current) as it:
                        for entry in it:
                            if entry.is_file(follow_symlinks=False):
                                # Store string path instead of Path object
                                files_batch.append(entry.path)
                                total_discovered += 1
                                discover_pbar.update(1)
                                
                                # Process batch when it reaches the size limit
                                if len(files_batch) >= batch_size:
                                    # Submit batch for deletion
                                    list(exe.map(fast_delete, files_batch, chunksize=500))
                                    total_deleted += len(files_batch)
                                    delete_pbar.update(len(files_batch))
                                    files_batch = []  # Clear batch
                                    
                            elif entry.is_dir(follow_symlinks=False):
                                stack.append(Path(entry.path))
                except PermissionError:
                    continue
            
            # Process remaining files in final batch
            if files_batch:
                list(exe.map(fast_delete, files_batch, chunksize=500))
                total_deleted += len(files_batch)
                delete_pbar.update(len(files_batch))
 
    print(f"\nTotal files discovered: {total_discovered}")
    print(f"Total files deleted: {total_deleted}")
    print("File deletion complete.")
     
    # Remove empty directories
    print("Removing empty directories...")
    for root, dirs, _ in os.walk(directory, topdown=False):
        for d in dirs:
            try:
                os.rmdir(Path(root) / d)
            except:
                pass
 
    print("Directory cleanup finished.")
 
# Run with optimized settings 
delete_directory_fast( 
    directory=r'G:\Thesis\ImageRetrieval\Professions_20k_DELETE', 
    workers=64,
    batch_size=50000  # Process in chunks to avoid memory issues
)

Deleting all files in: G:\Thesis\ImageRetrieval\Professions_20k_DELETE
Using 64 workers with batch size 50000...


Discovering: 26955files [00:00, 267419.47files/s]

A - deque([WindowsPath('G:/Thesis/ImageRetrieval/Professions_20k_DELETE')])
B
A - deque([WindowsPath('G:/Thesis/ImageRetrieval/Professions_20k_DELETE/Female_Accountant'), WindowsPath('G:/Thesis/ImageRetrieval/Professions_20k_DELETE/Male_Accountant')])
B
A - deque([WindowsPath('G:/Thesis/ImageRetrieval/Professions_20k_DELETE/Female_Accountant')])
B
C - G:\Thesis\ImageRetrieval\Professions_20k_DELETE\Female_Accountant\0.150_0003_14032147.jpg
C - G:\Thesis\ImageRetrieval\Professions_20k_DELETE\Female_Accountant\0.150_0003_14032598.jpg
C - G:\Thesis\ImageRetrieval\Professions_20k_DELETE\Female_Accountant\0.150_0003_14033201.jpg
C - G:\Thesis\ImageRetrieval\Professions_20k_DELETE\Female_Accountant\0.150_0003_1403477.jpg
C - G:\Thesis\ImageRetrieval\Professions_20k_DELETE\Female_Accountant\0.150_0003_14036619.jpg
C - G:\Thesis\ImageRetrieval\Professions_20k_DELETE\Female_Accountant\0.150_0003_14037187.jpg
C - G:\Thesis\ImageRetrieval\Professions_20k_DELETE\Female_Accountant\0.150_0003_140412

In [ ]:
# import os
# import ctypes
# import concurrent.futures
# from pathlib import Path
# from tqdm import tqdm

# # -------------------------------
# # Windows API delete (fast)
# # -------------------------------
# DeleteFileW = ctypes.windll.kernel32.DeleteFileW
# DeleteFileW.argtypes = [ctypes.c_wchar_p]
# DeleteFileW.restype = ctypes.c_bool

# SetFileAttributesW = ctypes.windll.kernel32.SetFileAttributesW
# SetFileAttributesW.argtypes = [ctypes.c_wchar_p, ctypes.c_uint32]
# SetFileAttributesW.restype = ctypes.c_bool

# FILE_ATTRIBUTE_NORMAL = 0x80


# def fast_delete(path: Path):
#     try:
#         p = str(path)
#         SetFileAttributesW(p, FILE_ATTRIBUTE_NORMAL)
#         DeleteFileW(p)
#     except Exception:
#         try:
#             os.remove(path)
#         except Exception:
#             pass


# # -------------------------------
# # Chunked delete directory
# # -------------------------------
# def delete_directory_fast(directory: str, workers: int = 6, batch_size: int = 1_000_000):
#     directory = Path(directory)

#     if not directory.exists():
#         print(f"Directory does not exist: {directory}")
#         return

#     print(f"Deleting all files in: {directory}")
#     print(f"Workers: {workers}")
#     print(f"Batch size: {batch_size:,}")

#     stack = [directory]
#     files = []

#     with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as exe:
#         with tqdm(desc="Files deleted", unit="files", mininterval=0.0) as pbar:

#             while stack:
#                 current = stack.pop()

#                 try:
#                     with os.scandir(current) as it:
#                         for entry in it:
#                             if entry.is_file(follow_symlinks=False):
#                                 files.append(entry.path)

#                                 # ---------- DELETE BATCH ----------
#                                 if len(files) >= batch_size:
#                                     for _ in exe.map(fast_delete, files, chunksize=500):
#                                         pbar.update(1)
#                                     files.clear()

#                             elif entry.is_dir(follow_symlinks=False):
#                                 stack.append(entry.path)

#                 except PermissionError:
#                     pass

#             # ---------- FINAL FLUSH ----------
#             if files:
#                 for _ in exe.map(fast_delete, files, chunksize=500):
#                     pbar.update(1)
#                 files.clear()

#     # -------------------------------
#     # Remove empty directories
#     # -------------------------------
#     print("Removing empty directories...")
#     for root, dirs, _ in os.walk(directory, topdown=False):
#         for d in dirs:
#             try:
#                 os.rmdir(Path(root) / d)
#             except Exception:
#                 pass

#     print("Directory cleanup finished.")


In [ ]:
# Run with SSD-optimized settings
delete_directory_fast(
    directory=r'G:\Thesis\ImageRetrieval\Professions_20k_DELETE',
    workers=64  # High worker count for SSD parallel I/O
)

Deleting all files in: G:\Thesis\ImageRetrieval\Professions_20k_DELETE
Workers: 64
Batch size: 1,000,000


Files deleted: 631367files [13:25:03, 939.02files/s] 